<a href="https://colab.research.google.com/github/devmurarijay13/pytorch-deep-learning/blob/main/pytorch_cust_churn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torchinfo

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torchinfo import summary

In [ ]:
df = pd.read_csv('/content/Bank_Churn.csv')

In [ ]:
df.head()

,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CustomerId       10000 non-null  int64  
 1   Surname          10000 non-null  object 
 2   CreditScore      10000 non-null  int64  
 3   Geography        10000 non-null  object 
 4   Gender           10000 non-null  object 
 5   Age              10000 non-null  int64  
 6   Tenure           10000 non-null  int64  
 7   Balance          10000 non-null  float64
 8   NumOfProducts    10000 non-null  int64  
 9   HasCrCard        10000 non-null  int64  
 10  IsActiveMember   10000 non-null  int64  
 11  EstimatedSalary  10000 non-null  float64
 12  Exited           10000 non-null  int64  
dtypes: float64(2), int64(8), object(3)
memory usage: 1015.8+ KB


In [ ]:
df.isna().sum()

,0
CustomerId,0
Surname,0
CreditScore,0
Geography,0
Gender,0
Age,0
Tenure,0
Balance,0
NumOfProducts,0
HasCrCard,0


In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
df.drop(columns=['CustomerId','Surname'],inplace=True)

In [ ]:
X = df.drop(columns='Exited')
y = df['Exited']

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y,random_state=42,test_size=0.2)

In [ ]:
scaler = StandardScaler()
gender_encoder = OneHotEncoder(sparse_output=False,drop='first')
geo_encoder = OneHotEncoder(sparse_output=False,drop='first')

In [ ]:
X_train

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
9254,686,France,Male,32,6,0.00,2,1,1,179093.26
1561,632,Germany,Male,42,4,119624.60,2,1,1,195978.86
1670,559,Spain,Male,24,3,114739.92,1,1,0,85891.02
6087,561,France,Female,27,9,135637.00,1,1,0,153080.40
6669,517,France,Male,56,9,142147.32,1,0,0,39488.04
...,...,...,...,...,...,...,...,...,...,...
5734,768,France,Male,54,8,69712.74,1,1,1,69381.05
5191,682,France,Female,58,1,0.00,1,1,1,706.50
5390,735,France,Female,38,1,0.00,3,0,0,92220.12
860,667,France,Male,43,8,190227.46,1,1,0,97508.04


### Data Preprocessing

In [ ]:
X_train['Geography'] = geo_encoder.fit_transform(X_train[['Geography']])
X_train['Gender'] = gender_encoder.fit_transform(X_train[['Gender']])

X_test['Geography'] = geo_encoder.transform(X_test[['Geography']])
X_test['Gender'] = gender_encoder.transform(X_test[['Gender']])

In [ ]:
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
X_train_tensor = torch.from_numpy(X_train)
X_test_tensor = torch.from_numpy(X_test)
y_train_tensor = torch.from_numpy(np.array(y_train))
y_test_tensor = torch.from_numpy(np.array(y_test))

### Dataset and DataLoader

In [ ]:
from torch.utils.data import Dataset,DataLoader

In [ ]:
class CustomDataset(Dataset):
    def __init__(self,features,labels):
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.features)

    def __getitem__(self,index):
        return self.features[index], self.labels[index]

In [ ]:
train_dataset = CustomDataset(X_train_tensor,y_train_tensor)
test_dataset = CustomDataset(X_test_tensor,y_test_tensor)

In [ ]:
train_dataset[1]

(tensor([-0.2039,  1.7257,  0.9132,  0.2949, -0.3484,  0.6968,  0.8084,  0.6492,
          0.9748,  1.6613], dtype=torch.float64),
 tensor(0))

In [ ]:
train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=32,shuffle=True)

### Model

In [ ]:
class Model(nn.Module):
    def __init__(self,n_dims):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(n_dims,3),
            nn.ReLU(),
            nn.Linear(3,1),
            nn.Sigmoid()
        )

    def forward(self,features):
        out = self.network(features)
        return out

In [ ]:
learning_rate = 0.1
epochs = 25

In [ ]:
model = Model(X_train_tensor.shape[1]).double()

In [ ]:
optimizer = torch.optim.SGD(model.parameters(),lr=learning_rate)

loss_function = nn.BCELoss()

In [ ]:
summary(model)

Layer (type:depth-idx)                   Param #
Model                                    --
├─Sequential: 1-1                        --
│    └─Linear: 2-1                       33
│    └─ReLU: 2-2                         --
│    └─Linear: 2-3                       4
│    └─Sigmoid: 2-4                      --
Total params: 37
Trainable params: 37
Non-trainable params: 0

### Training Pipeline

In [ ]:
for epoch in range(epochs):

    for batch_features, batch_labels in train_loader:
        ##fwrd pass
        y_pred = model(batch_features)

        ##loss
        loss = loss_function(y_pred,batch_labels.view(-1,1).double())

        ### clear gradients
        optimizer.zero_grad()

        ##backward pass
        loss.backward()

        ### parameter updation
        optimizer.step()

    print(f"Epoch {epoch + 1} , Loss : {loss.item()}")

Epoch 1 , Loss : 0.30885879596881227
Epoch 2 , Loss : 0.4026401629894477
Epoch 3 , Loss : 0.2621785064751336
Epoch 4 , Loss : 0.24369408439385426
Epoch 5 , Loss : 0.4135259113572139
Epoch 6 , Loss : 0.25968545658932174
Epoch 7 , Loss : 0.4377823064640833
Epoch 8 , Loss : 0.2964076343312718
Epoch 9 , Loss : 0.2873170181003839
Epoch 10 , Loss : 0.32508051330837967
Epoch 11 , Loss : 0.35164820189760443
Epoch 12 , Loss : 0.25400976955572224
Epoch 13 , Loss : 0.45359763294834965
Epoch 14 , Loss : 0.26295544222994893
Epoch 15 , Loss : 0.36856577966299775
Epoch 16 , Loss : 0.4232329468542956
Epoch 17 , Loss : 0.3293040177972753
Epoch 18 , Loss : 0.24363260942684548
Epoch 19 , Loss : 0.2960202755830353
Epoch 20 , Loss : 0.18550933890028842
Epoch 21 , Loss : 0.19508603708681538
Epoch 22 , Loss : 0.2934816020783868
Epoch 23 , Loss : 0.23141147340376444
Epoch 24 , Loss : 0.2942489632017631
Epoch 25 , Loss : 0.2429188913225088


### Model Evaluation

In [ ]:
model.eval() ### set the model to evaluation mode
accuracy_list = []

with torch.no_grad():

    for batch_features, batch_labels in test_loader:

        y_pred = model(batch_features)
        y_pred = (y_pred > 0.85).float()

        batch_accuracy = (y_pred.view(-1) == batch_labels).float().mean().item()
        accuracy_list.append(batch_accuracy)

overall_accuracy = sum(accuracy_list) / len(accuracy_list)
print(f'Accuracy : {overall_accuracy:.4f}')

Accuracy : 0.8338
